# Movie KV cache pre-generation (Qwen2.5-0.5B, reviews tail 500)

This notebook calls **`pregenerate_movie_kv_caches.py`** with **`--run-tags`** so each Kaggle session should use **exactly one** method tag to avoid GPU timeouts. Run **four separate sessions** for `ea`, `kvzip`, `finch_no_cpt`, and `finch_with_cpt` if you need all caches.

## What gets generated

- **Data**: `reviews_1000.csv` (Kaggle **Add Data**), last **`TAIL`** rows (default **500**).
- **Model**: `Qwen/Qwen2.5-0.5B-Instruct`.
- **Ratios (full run)**: `0.2`, `0.4`, `0.6`, `0.8` → **2000** `.pt` files per method (500 × 4).
- **Layout**: `{CACHE_DIR}/{sanitized_model}/{RUN_TAG}/comp{ratio}/cache_entry_*.pt`.

## Checkpoint resume

`CHECKPOINT_CSV` and caches live under **`/kaggle/working`** by default. A row with **`status=ok`** for **`(run_tag, compression_ratio)`** skips that **entire batch** on the next run. Individual `.pt` files still cause **per-row skip** inside a batch. Download **Output** before the session ends if you need to keep files.

## Prerequisites

1. **GPU** notebook; enable **Internet** (needed for `git clone` and the Hugging Face model).
2. **Add Data**: dataset containing `reviews_1000.csv`.
3. The config cell uses **`CLONE_REPO = True`**: the notebook clones from **`BENCH_URL`** into `/kaggle/working/kv-compression-benchmark`. **Your GitHub default branch must contain** `benchmarks/kv_cache_pregen/pregenerate_movie_kv_caches.py` (and `cache_io.py` in the same folder): commit and **push** from your PC before running this notebook, or the clone will be incomplete.
4. **Secrets**: **`HF_TOKEN`** (Hugging Face read token).

## Downstream note

Prefill text is `Review:` + review body plus the fixed sentiment task (and Finch delimiter/window layout). Whether this matches a separate “question outside cache” decode setup depends on how you load caches downstream.

In [ ]:
!pip install -q "kvpress>=0.2" "transformers>=4.45" accelerate pandas tqdm safetensors sentencepiece protobuf

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

# Hugging Face token (Kaggle Secrets)
try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            t = _sec.get_secret(name)
            if t:
                os.environ["HF_TOKEN"] = t
                os.environ["HUGGING_FACE_HUB_TOKEN"] = t
                print("Loaded secret:", name)
                break
        except Exception:
            continue
except Exception as e:
    print("Secrets:", e)

# All outputs under /kaggle/working — download from the Output tab before the session ends.
base = Path("/kaggle/working/movie_kv_pregen_qwen05b")
CACHE_DIR = base / "caches"
CHECKPOINT_CSV = base / "movie_kv_pregen_checkpoint.csv"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)
print("Working dir:", base.resolve())

# Shallow-clone the benchmark repo (requires Internet on the notebook).
CLONE_REPO = True
BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
# If the script lives on another branch, set e.g. GIT_BRANCH = "main"
GIT_BRANCH: str | None = None
WORK_ROOT = Path("/kaggle/working")

# --- Single method per session (required) ---
RUN_TAG = "ea"  # one of: ea | kvzip | finch_no_cpt | finch_with_cpt
_allowed = {"ea", "kvzip", "finch_no_cpt", "finch_with_cpt"}
if RUN_TAG not in _allowed:
    raise ValueError(f"RUN_TAG must be one of {_allowed}, got {RUN_TAG!r}")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TAIL = 500
RATIOS = ["0.2", "0.4", "0.6", "0.8"]

SMOKE_MODE = False
SMOKE_MAX_ROWS = 2
SMOKE_RATIOS = ["0.2"]

print("RUN_TAG:", RUN_TAG)
print("CHECKPOINT_CSV:", CHECKPOINT_CSV.resolve())
print("CACHE_DIR:", CACHE_DIR.resolve())
print("CLONE_REPO:", CLONE_REPO)

In [ ]:
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

MARKER = Path("benchmarks/kv_cache_pregen/pregenerate_movie_kv_caches.py")
TEXT_PATCH = Path("benchmarks/kv_cache_pregen/text_kvpress_patch.py")


def _repo_clone_complete(dest: Path) -> bool:
    return (dest / MARKER).is_file() and (dest / TEXT_PATCH).is_file()


def find_reviews_csv() -> Path:
    for p in Path("/kaggle/input").rglob("reviews_1000.csv"):
        return p
    raise FileNotFoundError(
        "Add a Kaggle dataset that contains reviews_1000.csv under /kaggle/input."
    )


def find_repo_root() -> Path:
    if CLONE_REPO:
        dest = WORK_ROOT / "kv-compression-benchmark"
        if _repo_clone_complete(dest):
            return dest
        # Stale or partial clone (e.g. old tree without text_kvpress_patch.py).
        if dest.exists():
            shutil.rmtree(dest)
        clone_cmd = [
            "git",
            "clone",
            "--depth",
            "1",
        ]
        if GIT_BRANCH:
            clone_cmd += ["--branch", GIT_BRANCH]
        clone_cmd += [BENCH_URL, str(dest)]
        subprocess.check_call(clone_cmd)
        if not (dest / MARKER).is_file():
            raise FileNotFoundError(
                f"After git clone, expected file missing:\n  {dest / MARKER}\n"
                f"Push commits that include this path to {BENCH_URL} (default branch), "
                f"or set GIT_BRANCH to the branch that has it, then re-run this cell."
            )
        return dest
    roots = list(Path("/kaggle/input").iterdir())
    for root in roots:
        if (root / MARKER).is_file():
            return root
        if root.is_dir():
            for sub in root.iterdir():
                if sub.is_dir() and (sub / MARKER).is_file():
                    return sub
    raise FileNotFoundError(
        "Repository not found. Add a dataset with this repo (or a zip containing "
        "benchmarks/kv_cache_pregen/), or set CLONE_REPO=True with Internet enabled."
    )


CSV_PATH = find_reviews_csv()
REPO_ROOT = find_repo_root()
SCRIPT = REPO_ROOT / MARKER

if not SCRIPT.is_file():
    raise FileNotFoundError(f"Script not found at {SCRIPT}")

print("CSV:", CSV_PATH)
print("REPO_ROOT:", REPO_ROOT)
print("SCRIPT:", SCRIPT, "(exists)")

## Smoke test (optional)

Set **`SMOKE_MODE = True`** in the configuration cell, then run this cell before the full run. It processes only **`SMOKE_MAX_ROWS`** rows and **`SMOKE_RATIOS`**, then checks that at least one `.pt` exists under the expected directory and loads with `torch.load`.

**Important:** the smoke command does **not** pass `--checkpoint-csv`. If it did, a successful 2-row smoke would mark `(RUN_TAG, ratio)` as finished and the **full run would skip that ratio**.

In [ ]:
from __future__ import annotations

import importlib.util
import shlex
import subprocess

import torch

if not SMOKE_MODE:
    print("SMOKE_MODE is False - skipping smoke cell.")
else:
    _model_safe = MODEL.replace("/", "_").replace(":", "_")

    cmd = [
        sys.executable,
        str(SCRIPT),
        "--csv",
        str(CSV_PATH),
        "--cache-dir",
        str(CACHE_DIR),
        "--model",
        MODEL,
        "--tail",
        str(int(TAIL)),
        "--max-rows",
        str(int(SMOKE_MAX_ROWS)),
        "--compression-ratio",
        *SMOKE_RATIOS,
        "--run-tags",
        RUN_TAG,
        "--device-map",
        "auto",
        "--bf16",
        "--no-flash-attn",
        # Do not pass --checkpoint-csv here (would block the full run for this ratio).
    ]
    if RUN_TAG in ("ea", "kvzip"):
        cmd.append("--kvzip-patch")

    print("Smoke command:", shlex.join(cmd))
    subprocess.check_call(cmd)

    _spec = importlib.util.spec_from_file_location("cache_io", str(SCRIPT.parent / "cache_io.py"))
    _cio = importlib.util.module_from_spec(_spec)
    assert _spec and _spec.loader
    _spec.loader.exec_module(_cio)

    for _r in SMOKE_RATIOS:
        _comp_lbl = _cio.compression_tag(float(_r))
        out_dir = CACHE_DIR / _model_safe / RUN_TAG / f"comp{_comp_lbl}"
        pts = list(out_dir.glob("cache_entry_*.pt"))
        assert pts, f"No cache_entry_*.pt under {out_dir}"
        try:
            _obj = torch.load(pts[0], map_location="cpu", weights_only=False)
        except TypeError:
            _obj = torch.load(pts[0], map_location="cpu")
        _has_layers = getattr(_obj, "layers", None) is not None
        print("Smoke OK:", out_dir.name, pts[0].name, type(_obj), "has_layers=", _has_layers)

## Full run

Set **`SMOKE_MODE = False`** in the configuration cell, then run this cell. It uses **`--tail 500`** and all four ratios. The same **`CHECKPOINT_CSV`** is shared across the four method tags if you reuse **`CACHE_DIR`**.

In [ ]:
import shlex
import subprocess

if SMOKE_MODE:
    raise RuntimeError(
        "Set SMOKE_MODE=False in the configuration cell before the full run "
        "(or do not execute this cell during smoke)."
    )

cmd = [
    sys.executable,
    str(SCRIPT),
    "--csv",
    str(CSV_PATH),
    "--cache-dir",
    str(CACHE_DIR),
    "--model",
    MODEL,
    "--tail",
    str(int(TAIL)),
    "--compression-ratio",
    *RATIOS,
    "--run-tags",
    RUN_TAG,
    "--device-map",
    "auto",
    "--bf16",
    "--no-flash-attn",
    "--checkpoint-csv",
    str(CHECKPOINT_CSV),
]
if RUN_TAG in ("ea", "kvzip"):
    cmd.append("--kvzip-patch")

print("Full command:", shlex.join(cmd))
subprocess.check_call(cmd)

## Optional: zip caches + checkpoint for download

On Kaggle, `/kaggle/working` can be downloaded from the **Output** tab after the run.

In [ ]:
# Uncomment to create an archive under /kaggle/working
# import shutil
# from pathlib import Path
# zname = Path("/kaggle/working/movie_kv_qwen05b_caches.zip")
# shutil.make_archive(str(zname.with_suffix("")), "zip", root_dir=str(base))
# print("Wrote", zname)